# 医療機関ホームページURL 本番実行（97,000件）

**作業データをGoogleドライブに保存するので、接続が切れても続きから再開できます。**
切れたときは、セル1から順に押し直すだけです。処理済みのぶんは自動で飛ばされます。

所要時間の目安は合計6〜10時間です。

## 1. ドライブに接続してプログラムを取得

「Googleドライブへのアクセスを許可しますか」と聞かれたら、
ご自身のアカウントを選んで「許可」を押してください。

**再開するときも、まずこのセルから実行してください。**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, shutil

WORK = "/content/drive/MyDrive/medical_url"      # 作業データの保存先（消さないでください）
REPO = "/content/search-web"
BRANCH = "claude/medical-facility-homepage-urls-v8z8e9"

os.makedirs(WORK, exist_ok=True)

if os.path.isdir(REPO):
    subprocess.run("git fetch origin", shell=True, cwd=REPO)
    subprocess.run(f"git reset --hard origin/{BRANCH}", shell=True, cwd=REPO)
else:
    subprocess.run(
        f"git clone -b {BRANCH} https://github.com/smasuzoe-jpg/search-web.git {REPO}",
        shell=True)

subprocess.run(f"pip install -q -r {REPO}/pipeline/requirements.txt", shell=True)

# 速度設定。照合の同時実行数を上げて所要時間を短縮する。
os.environ["VERIFY_CONCURRENCY"] = "24"    # サイトを同時に開く本数
os.environ["SEARCH_CONCURRENCY"] = "8"     # 検索を同時に投げる本数
os.environ["SEARCH_QPS"] = "10"            # 全体で毎秒10件を超えないようにする

print("準備できました")
print("作業フォルダ:", WORK)
for f in sorted(os.listdir(WORK)):
    print("   既存ファイル:", f)

## 2. APIキーを読み込む

左端の鍵アイコンで `SERPER_API_KEY` を登録済みであれば、押すだけで通ります。
実際に1件検索して、キーが使えるかその場で確かめます。

In [ ]:
# --- 共通設定（このセル単体でも動くように毎回定義する）---
import os
WORK = "/content/drive/MyDrive/medical_url"
REPO = "/content/search-web"
BRANCH = "claude/medical-facility-homepage-urls-v8z8e9"
os.environ["VERIFY_CONCURRENCY"] = "24"
os.environ["SEARCH_CONCURRENCY"] = "8"
os.environ["SEARCH_QPS"] = "10"

if not os.path.isdir(WORK) or not os.path.isdir(REPO):
    raise SystemExit(
        "先にセル1を実行してください。\n"
        "  ドライブの接続とプログラムの取得がまだ済んでいません。")

import os, requests
from google.colab import userdata

os.environ["SERPER_API_KEY"] = (userdata.get("SERPER_API_KEY") or "").strip()
key = os.environ["SERPER_API_KEY"]

r = requests.post("https://google.serper.dev/search",
                  headers={"X-API-KEY": key, "Content-Type": "application/json"},
                  json={"q": "護国寺内科・循環器クリニック 文京区 公式サイト",
                        "gl": "jp", "hl": "ja", "num": 3}, timeout=20)
if r.status_code == 200:
    print("✓ キーは正常に使えます")
elif r.status_code in (401, 403):
    print("✗ キーが違います。serper.dev で確認してください。")
elif r.status_code == 429:
    print("✗ クレジットを使い切っています。serper.dev で残数を確認してください。")
else:
    print(f"✗ 想定外の応答 HTTP {r.status_code}: {r.text[:200]}")

## 3. スプレッドシート3つを読み込む

1回目だけ実行してください。2回目以降はドライブに保存済みなので自動で飛ばされます。
3ファイル合計で5分ほどかかります。

In [ ]:
# --- 共通設定（このセル単体でも動くように毎回定義する）---
import os
WORK = "/content/drive/MyDrive/medical_url"
REPO = "/content/search-web"
BRANCH = "claude/medical-facility-homepage-urls-v8z8e9"
os.environ["VERIFY_CONCURRENCY"] = "24"
os.environ["SEARCH_CONCURRENCY"] = "8"
os.environ["SEARCH_QPS"] = "10"

if not os.path.isdir(WORK) or not os.path.isdir(REPO):
    raise SystemExit(
        "先にセル1を実行してください。\n"
        "  ドライブの接続とプログラムの取得がまだ済んでいません。")

from google.colab import auth
auth.authenticate_user()
import gspread, csv
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

SHEETS = {
    "list1": "https://docs.google.com/spreadsheets/d/1ZpqN1nu7q2D-Nl4KatbRtF9tb28uqEm-d5uqGi6Qyfk/edit",
    "list2": "https://docs.google.com/spreadsheets/d/1IuWhYT4l-U6tv9pvHAbnhrA19QVB12im0Q05H247N6w/edit",
    "list3": "https://docs.google.com/spreadsheets/d/1gtkXHcupWPhN-qmW0kepQeE6vBf4j4f04uwRo_g8tfo/edit",
}

total = 0
for name, url in SHEETS.items():
    dest = f"{WORK}/{name}.csv"
    if os.path.exists(dest):
        n = sum(1 for _ in open(dest, encoding="utf-8")) - 1
        print(f"{name}: 保存済み {n} 件（読み込みを飛ばします）")
        total += n
        continue
    header, rows = None, []
    for ws in gc.open_by_url(url).worksheets():
        vals = ws.get_all_values()
        if not vals:
            continue
        if header is None:
            header = vals[0]; rows.extend(vals)
        else:
            rows.extend(vals[1:] if vals[0] == header else vals)
    with open(dest, "w", encoding="utf-8", newline="") as f:
        csv.writer(f).writerows(rows)
    print(f"{name}: {len(rows)-1} 件を保存しました")
    total += len(rows) - 1

print(f"\n合計 {total} 件")

## 4. 本番実行

**このセルが本体です。6〜10時間かかります。**

3ファイルを順に、下ごしらえ → 検索 → 照合 → 出力まで通します。
進み具合が画面に流れます。

**接続が切れても大丈夫です。** 作業データはドライブに残るので、
セル1、セル2を押し直してからこのセルをもう一度押せば、続きから再開します。
検索済みのぶんは再検索されないので、クレジットも二重に消費しません。

In [ ]:
# --- 共通設定（このセル単体でも動くように毎回定義する）---
import os
WORK = "/content/drive/MyDrive/medical_url"
REPO = "/content/search-web"
BRANCH = "claude/medical-facility-homepage-urls-v8z8e9"
os.environ["VERIFY_CONCURRENCY"] = "24"
os.environ["SEARCH_CONCURRENCY"] = "8"
os.environ["SEARCH_QPS"] = "10"

if not os.path.isdir(WORK) or not os.path.isdir(REPO):
    raise SystemExit(
        "先にセル1を実行してください。\n"
        "  ドライブの接続とプログラムの取得がまだ済んでいません。")

import subprocess, sys, os, time

P = f"{REPO}/pipeline"

def run(cmd, stdout_path=None):
    """1コマンド実行。出力をそのまま画面に流す。"""
    print(f"\n$ {cmd}\n", flush=True)
    with open(stdout_path, "w", encoding="utf-8") if stdout_path else open(os.devnull, "w") as out:
        p = subprocess.Popen(cmd, shell=True, cwd=REPO,
                             stdout=out if stdout_path else subprocess.PIPE,
                             stderr=subprocess.PIPE, text=True, bufsize=1)
        for line in p.stderr:
            print(line, end="", flush=True)
        if not stdout_path and p.stdout:
            for line in p.stdout:
                print(line, end="", flush=True)
        p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"失敗しました（終了コード {p.returncode}）")

t0 = time.time()
for name in ["list1", "list2", "list3"]:
    src = f"{WORK}/{name}.csv"
    if not os.path.exists(src):
        print(f"{name}.csv がありません。セル3を実行してください。")
        continue
    print(f"\n{'='*60}\n{name} を処理します\n{'='*60}", flush=True)

    work = f"{WORK}/{name}_work.csv"
    if not os.path.exists(work):
        run(f"python3 {P}/01_prepare.py {src}", stdout_path=work)

    run(f"python3 {P}/03_search.py {work} {WORK}/{name}_cand.jsonl")
    run(f"python3 {P}/04_verify.py {WORK}/{name}_cand.jsonl {WORK}/{name}_verified.tsv")
    run(f"python3 {P}/05_export.py {work} {WORK}/{name}_verified.tsv {WORK}/{name}")

    hj = f"{WORK}/{name}_verified_details.jsonl"
    if os.path.exists(hj):
        run(f"python3 {P}/06_details.py {hj}", stdout_path=f"{WORK}/{name}_details.csv")
        run(f"python3 {P}/07_merge.py {WORK}/{name}_audit.csv {WORK}/{name}_details.csv",
            stdout_path=f"{WORK}/{name}_all.csv")

print(f"\n{'='*60}")
print(f"全部終わりました（所要 {(time.time()-t0)/3600:.1f} 時間）")
print(f"{'='*60}")

## 5. 進捗を見る

途中でも押せます。いまどこまで進んだかを表示します。

In [ ]:
# --- 共通設定（このセル単体でも動くように毎回定義する）---
import os
WORK = "/content/drive/MyDrive/medical_url"
REPO = "/content/search-web"
BRANCH = "claude/medical-facility-homepage-urls-v8z8e9"
os.environ["VERIFY_CONCURRENCY"] = "24"
os.environ["SEARCH_CONCURRENCY"] = "8"
os.environ["SEARCH_QPS"] = "10"

if not os.path.isdir(WORK) or not os.path.isdir(REPO):
    raise SystemExit(
        "先にセル1を実行してください。\n"
        "  ドライブの接続とプログラムの取得がまだ済んでいません。")

import os, csv
print(f"{'ファイル':8s} {'対象':>8s} {'検索済':>8s} {'照合済':>8s}")
print("-" * 40)
for name in ["list1", "list2", "list3"]:
    def count(path, header=False):
        if not os.path.exists(path):
            return 0
        n = sum(1 for _ in open(path, encoding="utf-8"))
        return max(0, n - 1) if header else n
    total = count(f"{WORK}/{name}_work.csv", header=True)
    cand = count(f"{WORK}/{name}_cand.jsonl")
    ver = count(f"{WORK}/{name}_verified.tsv", header=True)
    print(f"{name:8s} {total:8,d} {cand:8,d} {ver:8,d}")

## 6. 結果をまとめて取り出す

3ファイル分の結果を1つにまとめ、ドライブに保存してからダウンロードします。

- `final_all.csv` … **全43項目を1本にまとめたもの。まずこれを見てください**
- `final_import.csv` … レコードIDとURLだけ。HubSpotに戻す用（「高」のみ）
- `final_audit.csv` … 全項目つき。中身を確認する用
- `final_details.csv` … 診療時間と診療科目。医療機関コードで突き合わせて使います

In [ ]:
# --- 共通設定（このセル単体でも動くように毎回定義する）---
import os
WORK = "/content/drive/MyDrive/medical_url"
REPO = "/content/search-web"
BRANCH = "claude/medical-facility-homepage-urls-v8z8e9"
os.environ["VERIFY_CONCURRENCY"] = "24"
os.environ["SEARCH_CONCURRENCY"] = "8"
os.environ["SEARCH_QPS"] = "10"

if not os.path.isdir(WORK) or not os.path.isdir(REPO):
    raise SystemExit(
        "先にセル1を実行してください。\n"
        "  ドライブの接続とプログラムの取得がまだ済んでいません。")

import csv, glob, os

for kind in ["all", "import", "audit", "details"]:
    parts = [f"{WORK}/{n}_{kind}.csv" for n in ["list1", "list2", "list3"]]
    parts = [p for p in parts if os.path.exists(p)]
    if not parts:
        print(f"{kind}: まだ出力がありません")
        continue
    out = f"{WORK}/final_{kind}.csv"
    header, n = None, 0
    with open(out, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        for p in parts:
            with open(p, encoding="utf-8-sig") as g:
                rd = csv.reader(g)
                h = next(rd)
                if header is None:
                    header = h; w.writerow(h)
                for row in rd:
                    w.writerow(row); n += 1
    print(f"{kind}: {n:,} 件 -> {out}")

from google.colab import files
for kind in ["all", "import", "audit", "details"]:
    p = f"{WORK}/final_{kind}.csv"
    if os.path.exists(p):
        files.download(p)